# Lab 24: Securing an AI Security Agent

*AI Cybersecurity Masterclass · Lab 24 · AI60secLab*

**Companion video:** [▶ How an AI Security Agent Works in 60 Seconds](https://www.youtube.com/watch?v=MT3GFAZcNQk&list=PLG4l6Ssi9A00) · [▶ How to Secure AI Agents in 60 Seconds](https://www.youtube.com/watch?v=U9Hujhc8768&list=PLG4l6Ssi9A00)

**Research question:** How much authority should an AI agent have during an investigation, and what happens when it has too much?

A triage agent works real UNSW-NB15 alerts scored by a model. It can search logs, look up threat intelligence, draft reports, and propose containment (block an IP, isolate a host, disable an account). Some alerts carry attacker-controlled text. We compare three operating modes.

**Learning objectives**
- distinguish an agent (chooses the next step) from fixed automation (runs a playbook);
- apply least privilege through task profiles and tool risk tiers;
- validate tool arguments even for allowed tools;
- keep planning and authorization separate: an agent never grants itself permission;
- measure the cost of safety: human approvals requested.

> **About the simulated assistant.** These labs use a deterministic planner (`SOCAgent`), a deterministic stand-in for an LLM. It reproduces a documented failure of real models: it understands instructions written in many phrasings, including ones hidden in retrieved data, and proposes actions based on them. That keeps every result repeatable and free to run. The *controls* you build and measure are real code that could sit in front of a real model unchanged. What the simulation cannot tell you is how often a specific production model falls for a specific payload: that needs testing against that model.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.agent_security import (
    TOOL_RISK,
    TASK_PROFILES,
    PROTECTED_ASSETS,
    INJECTED_NOTE,
    Policy,
    alerts_from_scores,
    run_agent,
    summarize_run,
)

train, test = load_unsw(DATA_DIR)
train_sample = train.sample(40000, random_state=0)
test_sample = test.sample(3000, random_state=0).reset_index(drop=True)

X_train, y_train = split_xy(train_sample, "attack_cat")
X_test, _ = split_xy(test_sample, "attack_cat")
X_train, X_test = align_columns(X_train, X_test)
X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

model = random_forest_pipeline(X_train)
model.set_params(model__n_estimators=100)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)
classes = list(model.classes_)
normal = classes.index("Normal")
attack_score = 1 - proba[:, normal]
attack_proba = proba.copy()
attack_proba[:, normal] = -1
category_hint = np.array(classes)[attack_proba.argmax(axis=1)]

alerts = alerts_from_scores(test_sample, attack_score, category_hint, threshold=0.5)
print("Alerts:", len(alerts), "| real attacks among them:", sum(a.is_attack for a in alerts))
print("Alerts carrying injected text:", sum(bool(a.enrichment_note) for a in alerts))

## Step 1: Tools, risk tiers, and task profiles

In [ ]:
display(pd.Series(TOOL_RISK, name="risk tier").to_frame())
display(pd.Series({k: ", ".join(sorted(v)) for k, v in TASK_PROFILES.items()}, name="allowed tiers").to_frame())
print("Protected assets:", PROTECTED_ASSETS)
print("Attacker-controlled text in some alerts:", INJECTED_NOTE)

## Step 2: Three operating modes

- **fixed automation**: block every alert over 0.5, no context;
- **unsafe autonomy**: the agent reasons over the evidence and executes everything it proposes;
- **policy-gated agent**: the agent proposes, a deterministic policy decides, and a human approves containment.

In [ ]:
rows = []
for mode in ["fixed_automation", "unsafe_autonomy", "policy_gated"]:
    summary = summarize_run(run_agent(alerts, mode=mode))
    summary["mode"] = mode
    rows.append(summary)

modes = pd.DataFrame(rows).set_index("mode")
display(modes.T)

Fixed automation contains everything, including benign traffic and protected servers. The autonomous agent is smarter about containment, but it obeys instructions planted in alert text and grants itself admin. The policy-gated agent causes no harm, and pays for it in human approvals.

## Step 3: What the policy decided, and why

In [ ]:
gated = run_agent(alerts, mode="policy_gated")
display(gated.groupby(["tool", "decision"]).size().rename("steps").to_frame())
display(gated[gated["decision"] == "deny"]["policy_reason"].value_counts().rename("denials").to_frame())

## Step 4: The cost of safety

Every approval is analyst time. Raising the confidence floor for containment cuts approvals, but stops containment for attacks the model is less sure about. There is no free setting: choose it by risk and workload.

In [ ]:
rows = []
for floor in [0.90, 0.95, 0.98, 0.99]:
    log = run_agent(alerts, mode="policy_gated", policy=Policy(min_confidence_for_containment=floor))
    s = summarize_run(log)
    rows.append({
        "confidence_floor": floor,
        "human_approvals": s["human_approvals_requested"],
        "containment_on_attacks": s["containment_on_attacks"],
        "containment_on_benign": s["containment_on_benign"],
    })
display(pd.DataFrame(rows))

## Step 5: Least privilege by task

If the task is only triage, the agent should not even be able to propose containment.

In [ ]:
triage_only = run_agent(alerts, mode="policy_gated", policy=Policy(task_profile="triage"))
display(pd.Series(summarize_run(triage_only), name="triage-only profile").to_frame())

## Interpretation

1. Why is the autonomous agent's self-granted permission a design failure, not a model failure?
2. Which denials came from argument validation rather than tool permissions?
3. The policy-gated mode requested hundreds of approvals. What would you change first to reduce that load without adding risk?
4. When should an agent stop and ask instead of acting?

**Takeaway:** an agent is a model connected to capabilities. Give it the smallest set it needs, validate every argument, never let it authorize itself, and keep humans on high-impact decisions.